In [1]:
#Import packages
#---------------------------------------
import sys
import os
import glob
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import matplotlib
import kedm
import warnings
import gc
warnings.filterwarnings("ignore", category=RuntimeWarning) 

#Import your modules
#---------------------------------------
import CCM as cfn
import proc_functions as pfn

sys.path.insert(0, os.path.expanduser('~/Documents/admin_tools'))
from admin_tools import admin_functions as adfn

# Define paths
#----------------------------------------------------------------------
code = '~/Documents/empirical_dynamic_modelling'
data = '/snl/scratch25/dburrows/CCM'

%load_ext autoreload
sys.version

Kokkos::OpenMP::initialize WARNING: OMP_PROC_BIND environment variable not set
  In general, for best performance with OpenMP 4.0 or better set OMP_PROC_BIND=spread and OMP_PLACES=threads
  For best performance with OpenMP 3.1 set OMP_PROC_BIND=true
  For unit testing set OMP_PROC_BIND=false



'3.7.4 (default, Aug 13 2019, 20:35:49) \n[GCC 7.3.0]'

# Load in data

In [2]:
#group data
#================
#group by sample
xmap_l = np.asarray([os.path.basename(i) for i in np.sort(glob.glob(data + '/*xmap*'))])

import re
bln = xmap_l[[bool(re.search(r'(BLN|PTZ05)', i)) for i in xmap_l]]
ptz20 = xmap_l[[bool(re.search(r'(PTZ20)', i)) for i in xmap_l]]

#filter out 12 pre
s = pd.Series(ptz20)  # your series
mask = ~(s.str.contains(r'-12', na=False) & s.str.contains(r'pre', na=False))   # case-sensitive
ptz20 = ptz20[mask]

#unique sample list 
unq_l = np.append(np.unique(np.asarray([re.search(r'^(.*?_run-01)', i).group(1) for i in bln])),
          np.unique(np.asarray([''.join(re.search(r'^(.*?_run-01)(.{4})', i).groups()) for i in ptz20])))
unq_l

array(['PTZ-WILDTYPE-03_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-03_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-06_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-06_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-07_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-07_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-10_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-10_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-11_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-11_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-13_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-13_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-14_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-14_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-15_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-15_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-03_2photon_sess-01-6dpf_PTZ20_run-0

In [39]:
# mask data
%autoreload
#Compute fraction of cells that converge, and mean CCM in convergent cells
#==========================================================
from tqdm import tqdm

#parameters
n_int = 3 #expected interval files
int_l=[1.2, 1.4, 1.8] #expected intervals - make sure numbers are complete as in string! 
# -0.2 - i.e. allow a single negative value with <20% of the mean positive increase, to be convergent
ratio_val = -0.2 

results = []
# Loop through each unique base name
for curr in tqdm(unq_l, desc='Processing CCM files'):
    curr_l = glob.glob(f'{data}/{curr}*xmap*')
    og, mask, conv_arr = pfn.conv_check(curr_l=curr_l, n_int=n_int, int_l=int_l, ratio_val=ratio_val)
    np.save(f'{data}/{curr}_xmap_mask.npy', mask)

Processing CCM files: 100%|████████████████████████████████████████████████████████████████████████████████████████| 28/28 [06:57<00:00, 14.92s/it]


In [31]:
%autoreload
#load mask, trace & coords
#===============================
trace_l = np.sort(glob.glob(f'{data}/*regtrace*'))
xmap_l = np.sort(glob.glob(f'{data}/*_CCMxmap*'))
mask_l = np.sort(glob.glob(f'{data}/*mask*'))
coord_l = np.sort(glob.glob(f'{data}/*labcoord*'))

#load current into dic
dic = cfn.xtr(mask_l[0])
dic

{'name': 'PTZ-WILDTYPE-03_2photon_sess-01-6dpf_BLN_run-01',
 'trace': array([[1101.0016   ,  375.2699   ,  160.53424  , ...,  563.22595  ,
          127.223816 ,  208.07672  ],
        [ 454.70795  , 1038.5254   ,  936.5166   , ...,  417.1315   ,
          353.54297  ,  175.36679  ],
        [   1.2036324,  610.8864   ,   23.124939 , ...,   86.47761  ,
          347.2641   ,   28.184055 ],
        ...,
        [ 151.80298  ,   44.495804 ,    1.4848621, ...,   79.7669   ,
           19.013315 ,  224.81856  ],
        [  10.489203 ,    4.8878937,   81.60853  , ...,   20.048714 ,
           11.057799 ,   17.846321 ],
        [   9.782074 ,   33.105247 ,  475.08185  , ...,   32.403248 ,
          157.73413  ,   37.58297  ]], dtype=float32),
 'coord': array([['355.94736842105266', '435.0', '97.52631578947368',
         'Optic tectum - stratum periventriculare', 'Midbrain'],
        ['409.8125', '569.8125', '132.25', 'Rhombomere 3 - Dorsolateral',
         'Hindbrain'],
        ['459.375', '

In [ ]:
#directionality
#rows = each cell is the effects of the current row
#columns = each cell is the causes of the current column
#(ie. cell xy = to what extent does x cause y; cell yx = to what extent does y cause x). 

# Coarse network analysis

In [9]:
%autoreload
mask = dic['mask']
xmap = dic['xmap']
labels = dic['coord'][:,3]
regions, xmap_m, std_m, conv_m = cfn.ccm_region(mask=mask, xmap=xmap, labels=labels)

In [ ]:
from tqdm import tqdm
import pickle
#estimate rho and convergence fraction across all datasets
for i in tqdm(mask_l):
    dic = cfn.xtr(i)
    new_dic = {'name': dic['name']}
    mask = dic['mask']
    xmap = dic['xmap']
    labels = dic['coord'][:,3]
    regions, xmap_m, std_m, conv_m = cfn.ccm_region(mask=mask, xmap=xmap, labels=labels)
    new_dic.update({'granular': 
              {
               'regions': regions,
               'rho': xmap_m,
               'std': std_m,
               'conv': conv_m
              }

              })

    labels = dic['coord'][:,4]
    regions, xmap_m, std_m, conv_m = cfn.ccm_region(mask=mask, xmap=xmap, labels=labels)
    new_dic.update({'coarse': 
              {
               'regions': regions,
               'rho': xmap_m,
               'std': std_m,
               'conv': conv_m
              }

              })
    with open(f"{data}/{new_dic['name']}_region_stats.pkl", "wb") as f:
        pickle.dump(new_dic, f)

  0%|                                                                                                        | 0/28 [00:00<?, ?it/s]

In [ ]:
# generate heatmaps for each




In [ ]:
# more sophis

# Non-linearity & dimensionality

In [174]:
#Import packages
#---------------------------------------
import sys
import os
import glob
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import matplotlib
import kedm
import warnings
import gc
warnings.filterwarnings("ignore", category=RuntimeWarning) 

#Import your modules
#---------------------------------------
import CCM as cfn
import proc_functions as pfn

sys.path.insert(0, os.path.expanduser('~/Documents/admin_tools'))
from admin_tools import admin_functions as adfn

# Define paths
#----------------------------------------------------------------------
code = '~/Documents/empirical_dynamic_modelling'
data = '/snl/scratch25/dburrows/CCM'

%load_ext autoreload
sys.version

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


'3.7.4 (default, Aug 13 2019, 20:35:49) \n[GCC 7.3.0]'

# Load in data

In [2]:
#group data
#================
#group by sample
xmap_l = np.asarray([os.path.basename(i) for i in np.sort(glob.glob(data + '/*xmap*'))])

import re
bln = xmap_l[[bool(re.search(r'(BLN|PTZ05)', i)) for i in xmap_l]]
ptz20 = xmap_l[[bool(re.search(r'(PTZ20)', i)) for i in xmap_l]]

#filter out 12 pre
s = pd.Series(ptz20)  # your series
mask = ~(s.str.contains(r'-12', na=False) & s.str.contains(r'pre', na=False))   # case-sensitive
ptz20 = ptz20[mask]

#unique sample list 
unq_l = np.append(np.unique(np.asarray([re.search(r'^(.*?_run-01)', i).group(1) for i in bln])),
          np.unique(np.asarray([''.join(re.search(r'^(.*?_run-01)(.{4})', i).groups()) for i in ptz20])))
unq_l

array(['PTZ-WILDTYPE-03_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-03_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-06_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-06_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-07_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-07_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-10_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-10_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-11_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-11_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-13_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-13_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-14_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-14_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-15_2photon_sess-01-6dpf_BLN_run-01',
       'PTZ-WILDTYPE-15_2photon_sess-01-6dpf_PTZ05_run-01',
       'PTZ-WILDTYPE-03_2photon_sess-01-6dpf_PTZ20_run-0

In [39]:
# mask data
%autoreload
#Compute fraction of cells that converge, and mean CCM in convergent cells
#==========================================================
from tqdm import tqdm

#parameters
n_int = 3 #expected interval files
int_l=[1.2, 1.4, 1.8] #expected intervals - make sure numbers are complete as in string! 
# -0.2 - i.e. allow a single negative value with <20% of the mean positive increase, to be convergent
ratio_val = -0.2 

results = []
# Loop through each unique base name
for curr in tqdm(unq_l, desc='Processing CCM files'):
    curr_l = glob.glob(f'{data}/{curr}*xmap*')
    og, mask, conv_arr = pfn.conv_check(curr_l=curr_l, n_int=n_int, int_l=int_l, ratio_val=ratio_val)
    np.save(f'{data}/{curr}_xmap_mask.npy', mask)

Processing CCM files: 100%|████████████████████████████████████████████████████████████████████████████████████████| 28/28 [06:57<00:00, 14.92s/it]


In [26]:
%autoreload
#load mask, trace & coords
#===============================
trace_l = np.sort(glob.glob(f'{data}/*regtrace*'))
xmap_l = np.sort(glob.glob(f'{data}/*_CCMxmap*'))
mask_l = np.sort(glob.glob(f'{data}/*mask*'))
coord_l = np.sort(glob.glob(f'{data}/*labcoord*'))

#load current into dic
dic = cfn.xtr(mask_l[0])
dic

{'name': 'PTZ-WILDTYPE-03_2photon_sess-01-6dpf_BLN_run-01',
 'trace': array([[1101.0016   ,  375.2699   ,  160.53424  , ...,  563.22595  ,
          127.223816 ,  208.07672  ],
        [ 454.70795  , 1038.5254   ,  936.5166   , ...,  417.1315   ,
          353.54297  ,  175.36679  ],
        [   1.2036324,  610.8864   ,   23.124939 , ...,   86.47761  ,
          347.2641   ,   28.184055 ],
        ...,
        [ 151.80298  ,   44.495804 ,    1.4848621, ...,   79.7669   ,
           19.013315 ,  224.81856  ],
        [  10.489203 ,    4.8878937,   81.60853  , ...,   20.048714 ,
           11.057799 ,   17.846321 ],
        [   9.782074 ,   33.105247 ,  475.08185  , ...,   32.403248 ,
          157.73413  ,   37.58297  ]], dtype=float32),
 'coord': array([['355.94736842105266', '435.0', '97.52631578947368',
         'Optic tectum - stratum periventriculare', 'Midbrain'],
        ['409.8125', '569.8125', '132.25', 'Rhombomere 3 - Dorsolateral',
         'Hindbrain'],
        ['459.375', '

In [ ]:
#directionality
#rows = each cell is the effects of the current row
#columns = each cell is the causes of the current column
#(ie. cell xy = to what extent does x cause y; cell yx = to what extent does y cause x). 

# Coarse network analysis

In [28]:
%autoreload
mask = dic['mask']
xmap = dic['xmap']
labels = dic['coord'][:,3]
regions, xmap_m, std_m, conv_m = cfn.ccm_region(mask=mask, xmap=xmap, labels=labels)

In [29]:
dic

{'name': 'PTZ-WILDTYPE-03_2photon_sess-01-6dpf_BLN_run-01',
 'trace': array([[1101.0016   ,  375.2699   ,  160.53424  , ...,  563.22595  ,
          127.223816 ,  208.07672  ],
        [ 454.70795  , 1038.5254   ,  936.5166   , ...,  417.1315   ,
          353.54297  ,  175.36679  ],
        [   1.2036324,  610.8864   ,   23.124939 , ...,   86.47761  ,
          347.2641   ,   28.184055 ],
        ...,
        [ 151.80298  ,   44.495804 ,    1.4848621, ...,   79.7669   ,
           19.013315 ,  224.81856  ],
        [  10.489203 ,    4.8878937,   81.60853  , ...,   20.048714 ,
           11.057799 ,   17.846321 ],
        [   9.782074 ,   33.105247 ,  475.08185  , ...,   32.403248 ,
          157.73413  ,   37.58297  ]], dtype=float32),
 'coord': array([['355.94736842105266', '435.0', '97.52631578947368',
         'Optic tectum - stratum periventriculare', 'Midbrain'],
        ['409.8125', '569.8125', '132.25', 'Rhombomere 3 - Dorsolateral',
         'Hindbrain'],
        ['459.375', '

In [323]:
%autoreload
ccm_region?

# Non-linearity & dimensionality